# Field validation — `frontogenesis` (SURF pipeline)

End-to-end validation of every calculated field in the **frontogenesis** surface subset: the notebook RUNs the pipeline, LOADs its own output, and validates each field with dependency-chain maps, PDFs, and a literature comparison.

| | |
|---|---|
| Subset | `frontogenesis` (SURF) |
| Timestep | 2012-11-09 12:00:00 (`20121109_120000`) |
| run_id | `field_validation_v1` |
| Plan | `prompts/field_validation.md` |
| Field reference | `docs/Fields.md` |

Frontogenesis-function diagnostics.  Every computed step is shown: buoyancy-gradient components (db_dx, db_dy) and SSH-gradient components (dEta_dx, dEta_dy) are computed live and plotted; the velocity-Jacobian components feeding F(u,v) are identical to the ones validated in `kinematic.ipynb` (referenced, not re-plotted).

## Section 1 — RUN the SURF pipeline for this subset + timestep

One cell (pattern from
`notebooks/notebooks_global/running_generate_global_script.ipynb`).
Existing stores for this date are skipped unless `--clobber` is added,
so re-running is a cheap no-op.

In [ ]:
# Section 1: run the SURF pipeline for this subset and timestep.
SUBSET   = "frontogenesis"
PIPELINE = "SURF"
RUN_ID   = "field_validation_v1"
DATE     = "2012-11-09 12:00:00"   # single validation timestep

!generate-global \
    --config ../../../configs/global/run/field_validation_surface.yaml \
    --pipeline $PIPELINE \
    --subset $SUBSET \
    --run_id $RUN_ID

## Section 2 — LOAD the data generated in Section 1

One cell (pattern from
`notebooks/notebooks_global/assess_generate_global_script.ipynb`):
product reader + grid reader, with shape/date confirmation printout.

In [ ]:
# Section 2: load the store written in Section 1 + the shared grid.
import numpy as np
import matplotlib.pyplot as plt

import dbof.io.filesystems as filesystems
import dbof.global_dataset_creation.zarr_dataset_global as zarr_dataset
import dbof.global_dataset_creation.zarr_grid_global as zarr_grid
from dbof.global_dataset_creation.subset_definitions import (
    get_subset_definition,
)

S3_ENDPOINT = "https://s3-west.nrp-nautilus.io"
BUCKET      = "dbof"
FOLDER      = "surface_fields"       # SURF output folder
DATE_PREFIX = "20121109_120000"      # matches DATE in Section 1

defn = get_subset_definition(PIPELINE, SUBSET)
# get_subset_definition already folds per-pipeline extras (e.g.
# oceQnet for SURF) into model_data_feature_channels.
CHANNELS = (list(defn["model_data_feature_channels"])
            + list(defn["compute_features_channels"]))

fs, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
reader = zarr_dataset.GlobalZarrDatasetReader(
    bucket=BUCKET, folder=FOLDER, run_id=RUN_ID,
    dataset_name=defn["dataset_name"], date_prefix=DATE_PREFIX, fs=fs,
)

fs_grid, _ = filesystems.create_s3_filesystems(S3_ENDPOINT)
grid_reader = zarr_grid.GlobalGridZarrReader(
    bucket=BUCKET, folder="LLC4320_GRID_2D",
    dataset_name="llc4320_grid.zarr", fs=fs_grid,
)
XC, YC = grid_reader.lon, grid_reader.lat

print(reader)
print(f"channels  : {reader.channel_names}")
print(f"shape     : {reader.shape}  (C, H, W)")
print(f"iteration : {reader.iteration}")
print(f"grid      : XC {XC.shape}, "
      f"lon [{XC.min():.1f}, {XC.max():.1f}], "
      f"lat [{YC.min():.1f}, {YC.max():.1f}]")

## Section 3 — SUBSET: `frontogenesis`

Channels (verbatim from `subset_definitions.SURFACE_SUBSETS`):

computed — `frontogenesis_tendency`, `ug`, `vg`, `frontogenesis_geo`, `frontogenesis_ageo`, `Wstar`

In [ ]:
# Section 3: guard — store channels must match the code definition.
print(f"subset '{SUBSET}': {len(CHANNELS)} channels")
for ch in CHANNELS:
    print(f"  - {ch}")
assert set(reader.channel_names) == set(CHANNELS), (
    "store channels differ from subset_definitions — regenerate the "
    "store (Section 1, --clobber) or check the code definition")
print("OK: store channel set matches subset_definitions")

## Section 4 — Field & dependency table

| FIELD NAME | UNITS | EQUATION | DEPENDENCIES | LOCATION OF CALC IN CODE |
|---|---|---|---|---|
| frontogenesis_tendency | s⁻⁵ | F(u,v) = −[uₓ·bₓ² + (u_y + vₓ)·bₓ·b_y + v_y·b_y²] | J (U, V); db_dx, db_dy (← b ← Theta, Salt) | `calculate_fields.frontogenesis_tendency` (`_frontogenesis_formula`) |
| ug | m s⁻¹ | −(g/f)·∂η/∂y | dEta_dy, coriolis_f | `calculate_fields.geostrophic_velocity` |
| vg | m s⁻¹ | +(g/f)·∂η/∂x | dEta_dx, coriolis_f | `calculate_fields.geostrophic_velocity` |
| frontogenesis_geo | s⁻⁵ | F(ug,vg) — same formula with geostrophic velocity gradients | ug, vg gradients; db_dx, db_dy | `calculate_fields.frontogenesis_geo` |
| frontogenesis_ageo | s⁻⁵ | F(u,v) − F(ug,vg) | both tendencies | `surface_subsets.compute_frontogenesis` (inline) |
| Wstar | s⁻² | W* = 4·sgn(λ₂)·√(λ₁² + λ₂²); λ₂ = W/4 with W = σₙ² + σₛ² − ζ²; λ₁ = λ₂/2 + √(λ₂² + \|Q\|²/f²)/2; Q = −(uₓbₓ + vₓb_y, u_ybₓ + v_yb_y) (Bachman 2021) | J; db_dx, db_dy; f | `calculate_fields.modified_okubo_weiss` |

Live-plotted intermediates: raw `Eta`, rotated `U`/`V`, `buoyancy`,
`db_dx`/`db_dy` (`calculate_fields.compute_buoyancy_gradients`),
`dEta_dx`/`dEta_dy` (`native_gradient.
calculate_native_gradient_tracer`), and rect-grid `coriolis_f`.
J components → validated in `kinematic.ipynb` (identical arrays).

Processing operations: land masking; staggered→tracer interpolation
+ CS/SN rotation (U, V, J); native-grid differentiation (∇b, ∇η, J —
halo rim); f-division (ug, vg, W* — equatorial extremes expected);
face→lat-lon stitching; global downsampling.

### Raw inputs & intermediates computed live

The store holds only the six output channels; chains start from raw Eta, rotated velocities, buoyancy, and the buoyancy/SSH gradient components — computed here from the same OSN snapshot with the same code and batch-stitched.  coriolis_f is recomputed on the rect grid (independent cross-check).

In [ ]:
# Live raw inputs + intermediates (pipeline loaders/code).
import dbof.preprocessing.calculate_fields as calculate_fields
import dbof.utils.native_gradient as ng
from dbof.cli.generate_global import load_snapshot
from dbof.global_dataset_creation.data_sources import get_data_source
from dbof.global_dataset_creation.grid_setup import set_up_grid
from dbof.utils.faces_to_latlon import stitch_and_mask
from dbof.preprocessing.physical_constants import OMEGA_EARTH

ds_grid, land_mask, xgrid = set_up_grid(PIPELINE, None)
ds_raw, ds_merge, it = load_snapshot(
    PIPELINE, DATE, ds_grid, ["Theta", "Salt", "Eta", "U", "V"],
    surface_only=False, data_source=get_data_source(PIPELINE),
)
print(f"OSN iteration {it}  (store iteration {reader.iteration})")
for _v in ds_merge.data_vars:
    if ds_merge[_v].ndim >= 2:
        print(f"  raw {_v}: {ds_merge[_v].dtype}")

u_east, v_north = calculate_fields.geographic_velocity(
    ds_merge, xgrid)
bg = calculate_fields.compute_buoyancy_gradients(ds_merge, xgrid)
gE = ng.calculate_native_gradient_tracer(
    ds_merge.Eta, ds_merge, grid=xgrid)
J = calculate_fields.compute_velocity_jacobian(ds_merge, xgrid)

# Extra slice region for the Bachman et al. (2021) comparison.
EXTRA_REGIONS = ["kerguelen"]

live_map = {
    "Eta": ds_merge["Eta"], "U": u_east, "V": v_north,
    "buoyancy": calculate_fields.buoyancy_of_field(ds_merge),
    "db_dx": bg.zonal, "db_dy": bg.merid,
    "dEta_dx": gE[0], "dEta_dy": gE[1],
    "du_dx": J.du_dx, "du_dy": J.du_dy,
    "dv_dx": J.dv_dx, "dv_dy": J.dv_dy,
}
# Batch-stitch the live fields and slice to the validation domains
# immediately — at most BATCH full-res arrays alive at once.
from dbof.plotting import regions

SLICE_REGIONS = (regions.REGION_ORDER
                 + globals().get("EXTRA_REGIONS", []))
BATCH = 4
mask = {"_land_mask": (ds_merge.hFacC == 0)}
names = list(live_map)
region_arrays = {}
for i0 in range(0, len(names), BATCH):
    grp = names[i0:i0 + BATCH]
    ds_conv = ds_raw.assign({n: live_map[n] for n in grp})[grp]
    chw = stitch_and_mask(ds_conv, grp, mask)
    for k, n in enumerate(grp):
        region_arrays[n] = regions.select_all_regions(
            chw[k], XC, YC, names=SLICE_REGIONS)
    del chw
    print(f"stitched + sliced: {grp}")
print(f"live fields ready: {names}")

# coriolis_f on the rect grid (cross-check on the pipeline's f),
# ocean-masked via the sliced Eta NaN pattern per region.
f_full = (2.0 * OMEGA_EARTH
          * np.sin(np.radians(YC))).astype("float32")
region_arrays["coriolis_f"] = {}
for rname, (x, y, eta) in region_arrays["Eta"].items():
    jsl, isl = regions.region_index_slices(XC, YC, rname)
    f_sub = np.where(np.isfinite(eta), f_full[jsl, isl], np.nan)
    region_arrays["coriolis_f"][rname] = (x, y, f_sub)
del f_full
print("coriolis_f ready (rect-grid cross-check)")

In [ ]:
# Slice the STORE channels to the validation domains (live fields, if
# any, were sliced in the previous cell).  Full-res arrays released
# immediately after slicing.
from dbof.plotting import regions
from dbof.plotting.field_cmaps import load_field_cmaps

CMAP_CFG, DIVERGING = load_field_cmaps()

# region_arrays[field][region] = (x, y, arr)
region_arrays = globals().get("region_arrays", {})
SLICE_REGIONS = (regions.REGION_ORDER
                 + globals().get("EXTRA_REGIONS", []))
for ch in CHANNELS:
    arr = reader.get_channel_snapshot(ch)
    region_arrays[ch] = regions.select_all_regions(
        arr, XC, YC, names=SLICE_REGIONS)
    del arr

for ch in region_arrays:
    x, y, sub = region_arrays[ch]["gulf_stream"]
    print(f"{ch:24s} gulf_stream {sub.shape}  "
          f"min {np.nanmin(sub):.3g}  max {np.nanmax(sub):.3g}")

## Section 5 — Per-field validation

- **Figure 1 — maps**: columns = the field's full dependency chain
  (raw → components → final; every computed step is shown), rows =
  validation domains.  One shared colour scale per column; land/halo
  NaNs gray; regional boxes on the global row.
- **Figure 2 — PDFs**: same grid.  Probability density; land +
  halo-rim NaNs removed; bins shared per field across domains;
  Eq. Pacific row |lat|>2° filtered for f-normalised fields.
- **Literature comparisons** live in Section 6 at the end of the
  notebook — one subsection PER REFERENCE (a reference may validate
  several fields at once), only where a reference exists.  Images in
  `../literature_figures/`, named
  `{field(s)}_{Citation}_{description}.png`.

In [ ]:
# Section 5 helpers: one call per figure, shared by all fields.
from pathlib import Path

import cartopy.crs as ccrs

from dbof.plotting.global_maps import plot_global_field
from dbof.plotting.pipeline_grids import (
    pipeline_map_grid, mask_wrap_cells, LAND_COLOR,
)
from dbof.plotting.pdfs import pipeline_pdf_grid
from dbof.plotting.literature_comparison import side_by_side

# Flat literature directory; files named
# {field}_{Citation}_{description}.png
LIT_DIR = Path("../literature_figures")

# Full dependency chain per field (columns of Figures 1-2), including
# component-level intermediates (gradient / Jacobian components).
CHAINS = {
    "frontogenesis_tendency": ["U", "V", "du_dx", "du_dy", "dv_dx", "dv_dy", "db_dx", "db_dy", "frontogenesis_tendency"],
    "ug": ["Eta", "dEta_dy", "coriolis_f", "ug"],
    "vg": ["Eta", "dEta_dx", "coriolis_f", "vg"],
    "frontogenesis_geo": ["ug", "vg", "db_dx", "db_dy", "frontogenesis_geo"],
    "frontogenesis_ageo": ["frontogenesis_tendency", "frontogenesis_geo", "frontogenesis_ageo"],
    "Wstar": ["U", "V", "du_dx", "du_dy", "dv_dx", "dv_dy", "db_dx", "db_dy", "Wstar"],
}

# f-normalised fields: |lat|>2 deg filter on the Eq. Pacific row of
# the PDFs only (maps annotated instead) — plan Clarification 8.
F_NORM = {"Wstar", "frontogenesis_ageo", "frontogenesis_geo", "ug", "vg"}

# Fields drawn/binned on log scales (∝-squared fields).
LOG_FIELDS = set()

PDF_NOTE = ("PDFs: density; land+rim NaNs removed; shared bins "
            "across domains; log10-x for \u221d-squared fields")


def _pdf_arrays(field):
    """Region arrays for the PDF grid of one field.

    Applies the |lat|>2 deg filter to the Eq. Pacific row for
    f-normalised chain members (filter stated in the figure title).
    Inputs: field (str).  Outputs: dict like region_arrays.
    Generated by LH and Claude
    """
    out = {}
    for f in CHAINS[field]:
        d = dict(region_arrays[f])
        if f in F_NORM:
            x, y, a = d["eq_pacific"]
            d["eq_pacific"] = (x, y,
                               np.where(np.abs(y) > 2.0, a, np.nan))
        out[f] = d
    return out


def figure1_maps(field):
    """Figure 1: dependency-chain map grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    note = (" | f-normalised: Eq. Pacific extreme near equator "
            "(expected)" if field in F_NORM else "")
    pipeline_map_grid(
        CHAINS[field], region_arrays, CMAP_CFG,
        diverging_cmaps=DIVERGING, log_scale_channels=LOG_FIELDS,
        suptitle=f"Figure 1 \u2014 {field}: pipeline maps{note}",
    )
    plt.show()


def figure2_pdfs(field):
    """Figure 2: dependency-chain PDF grid for one field.

    Inputs: field (str) — output channel name (key into CHAINS).
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    note = (" | Eq. Pacific: |lat|>2\u00b0 filter (f-normalised)"
            if set(CHAINS[field]) & F_NORM else "")
    pipeline_pdf_grid(
        CHAINS[field], _pdf_arrays(field), CMAP_CFG,
        log10_fields=LOG_FIELDS,
        suptitle=f"Figure 2 \u2014 {field}: {PDF_NOTE}{note}",
    )
    plt.show()


def figure3_literature(field, png_name=None, caption=None,
                       global_view=False):
    """Figure 3: our data vs literature .png for one field.

    Inputs: field (str); png_name (str or None) — file in LIT_DIR;
    caption (str or None) — discussion text; global_view (bool) —
    render OUR panel as a global Robinson map (use when the
    literature figure is a global view) instead of the default
    Gulf Stream regional map.
    Outputs: displays the figure.
    Generated by LH and Claude
    """
    region = "global" if global_view else "gulf_stream"

    def _render(ax):
        x, y, arr = region_arrays[field][region]
        if global_view:
            # Same seam/Arctic handling as the Figure 1 global row.
            arr = mask_wrap_cells(x, y, arr)
        ax.set_facecolor(LAND_COLOR)
        im, label = plot_global_field(
            ax, x, y, arr, field, CMAP_CFG,
            log_scale_channels=LOG_FIELDS, diverging_cmaps=DIVERGING,
            transform=ccrs.PlateCarree() if global_view else None,
            add_coastline=global_view,
            coastline_kw={"linewidth": 0.4, "edgecolor": "k"},
        )
        if im is not None:
            plt.colorbar(im, ax=ax, orientation="horizontal",
                         fraction=0.04, pad=0.04, label=label)

    side_by_side(
        _render, LIT_DIR / png_name if png_name else None,
        projection=ccrs.Robinson() if global_view else None,
        caption=caption or ("Discussion: awaiting literature "
                            f"reference for {field}."),
    )
    plt.show()

### 5.1 frontogenesis_tendency — F(u,v) = −[uₓbₓ² + (u_y+vₓ)bₓb_y + v_yb_y²]

Kinematic frontogenesis function [s⁻⁵]: the rate of |∇b|² sharpening by the horizontal flow.  Positive filaments along strain-dominated fronts (Gulf Stream north wall, ACC).

In [ ]:
figure1_maps("frontogenesis_tendency")

In [ ]:
figure2_pdfs("frontogenesis_tendency")

### 5.2 ug (geostrophic U)

−(g/f)·dEta_dy.  Tracks rotated U at mesoscale away from the equator; equatorial blow-up expected.

In [ ]:
figure1_maps("ug")

In [ ]:
figure2_pdfs("ug")

### 5.3 vg (geostrophic V)

+(g/f)·dEta_dx.  Same checks, meridional component.

In [ ]:
figure1_maps("vg")

In [ ]:
figure2_pdfs("vg")

### 5.4 frontogenesis_geo — F(ug,vg), same formula with geostrophic gradients

Geostrophic contribution [s⁻⁵]: −[ugₓbₓ² + (ug_y+vgₓ)bₓb_y + vg_yb_y²]; smoother than the full tendency.

In [ ]:
figure1_maps("frontogenesis_geo")

In [ ]:
figure2_pdfs("frontogenesis_geo")

### 5.5 frontogenesis_ageo — F(u,v) − F(ug,vg)

Ageostrophic/submesoscale residual [s⁻⁵] (dispatcher-inline difference).

In [ ]:
figure1_maps("frontogenesis_ageo")

In [ ]:
figure2_pdfs("frontogenesis_ageo")

### 5.6 Wstar — W* = 4·sgn(λ₂)·√(λ₁² + λ₂²)

Modified Okubo-Weiss (Bachman 2021) [s⁻²]: λ₂ = W/4 (classical Okubo-Weiss), λ₁ = λ₂/2 + √(λ₂² + |Q|²/f²)/2 with the QG Q-vector Q = −(uₓbₓ + vₓb_y, u_ybₓ + v_yb_y).  Strain/vorticity partition sensitised to frontal ∇b; NaN-heavy at the equator (expected; PDF row filtered).

In [ ]:
figure1_maps("Wstar")

In [ ]:
figure2_pdfs("Wstar")

## Section 6 — Literature comparisons

One subsection per reference (a reference may validate several fields); only fields with published counterparts appear here.

### 6.1 Bachman et al. (2021) — W*, F, |∇b| over the Kerguelen Plateau

Reference region: 60–85°E, 50–38°S (``regions.REGIONS['kerguelen']``,
sliced in the live cell via ``EXTRA_REGIONS``).  Left column: our
snapshot (2012-11-09); right: Bachman et al. (2021) maps.  Note the
reference shows a sub-window (~74–85°E) of the region and a different
model/date — compare structure and magnitudes, not features.

In [ ]:
# Bachman et al. (2021) Fig-1-style comparison: stacked maps of
# Wstar, F (their Eq. 38), |grad b| over the Kerguelen Plateau,
# with COLOUR SCALES PINNED TO THE REFERENCE FIGURE.
import matplotlib.colors as mcolors

from dbof.plotting.literature_comparison import stacked_side_by_side

KREG = "kerguelen"


def _kerg(field):
    """Kerguelen slice of one field.  Generated by LH and Claude."""
    return region_arrays[field][KREG]


_xk, _yk, _dbx = _kerg("db_dx")
_dby = _kerg("db_dy")[2]
gradb_mag_kerg = np.hypot(_dbx, _dby)

# Bachman et al. (2021) Eq. 38: F = 2 Q.grad(b)/|grad(b)| [s-3].
# Our store channel frontogenesis_tendency IS Q.grad(b) (expand the
# dot product to see it equals _frontogenesis_formula), so their F
# is exactly 2*ours/|grad b|.
_F_store = _kerg("frontogenesis_tendency")[2]
F_b38 = np.where(gradb_mag_kerg > 0,
                 2.0 * _F_store / gradb_mag_kerg, np.nan)
_v = np.abs(F_b38[np.isfinite(F_b38)])
print("F (Eq. 38) |value| percentiles [s-3]: "
      + ", ".join(f"p{p}={np.percentile(_v, p):.2e}"
                  for p in (50, 90, 99)))
print("reference map scale: +/-5e-15")


def _kerg_panel(arr, cmap_field, norm, label):
    """Renderer with an explicit (literature-pinned) norm.

    Inputs: arr (2D); cmap_field (str, field_cmaps key for the
    colormap); norm (matplotlib Normalize); label (str).
    Outputs: callable(ax).  Generated by LH and Claude
    """
    def _render(ax):
        ax.set_facecolor(LAND_COLOR)
        im, _ = plot_global_field(
            ax, _xk, _yk, arr, cmap_field, CMAP_CFG,
            diverging_cmaps=DIVERGING, add_coastline=False,
            norm=norm)
        if im is not None:
            plt.colorbar(im, ax=ax, label=label, fraction=0.04,
                         extend="both")
        ax.set_xticks([])
        ax.set_yticks([])
    return _render


stacked_side_by_side(
    [_kerg_panel(_kerg("Wstar")[2], "Wstar",
                 mcolors.TwoSlopeNorm(vcenter=0, vmin=-1e-7,
                                      vmax=1e-7),
                 "W* (s⁻²)  [ref scale ±1e-7]"),
     _kerg_panel(F_b38,
                 "frontogenesis_tendency",
                 mcolors.TwoSlopeNorm(vcenter=0, vmin=-5e-15,
                                      vmax=5e-15),
                 "F = 2Q·∇b/|∇b| (s⁻³) "
                 "[ref scale ±5e-15]"),
     _kerg_panel(gradb_mag_kerg, "gradb_mag",
                 mcolors.Normalize(vmin=0, vmax=5e-7),
                 "|∇b| (s⁻²)  [ref scale "
                 "0–5e-7]")],
    LIT_DIR / ("Wstar-F-gradb_Bachman-etal(2021)_"
               "Kerguelen-maps.png"),
    our_titles=["Wstar", "F (Bachman Eq. 38)", "|∇b|"],
    caption=("Colour scales pinned to Bachman et al. (2021).  W* and "
             "|∇b| compare directly.  F computed per their "
             "Eq. 38: F = 2Q·∇b/|∇b| [s⁻"
             "³] = 2×(our Q·∇b channel)/"
             "|∇b|.  Reference fields are from an IDEALISED "
             "channel simulation (their year 5, day 100), not "
             "LLC4320 — compare morphology; residual "
             "magnitude offsets may be model/seasonal."),
)
plt.show()

### 6.2 The two F definitions side by side (same numerator)

Our product channel is **F = Q·∇b** [s⁻⁵] (unnormalised — this is
and remains the pipeline definition).  Bachman et al. (2021) Eq. 38
uses **F₃₈ = 2Q·∇b/|∇b|** [s⁻³] (Hoskins 1982).  Same numerator,
different normalisation: F emphasises sharpening where |∇b| is
already strong (|∇b|² weighting); F₃₈ measures the rate of change of
|∇b| itself.  Shown side by side over the Kerguelen box, each on its
own percentile scale.

In [ ]:
# Our F (store channel, unchanged) vs Bachman Eq. 38 — same
# numerator Q.grad(b), different normalisation.
fig, _axes = plt.subplots(1, 2, figsize=(15, 5.5))
for _ax, _arr, _ttl in [
    (_axes[0], _F_store,
     "ours: F = Q·∇b  (s⁻⁵)"),
    (_axes[1], F_b38,
     "Bachman Eq. 38: F₃₈ = 2Q·∇b/"
     "|∇b|  (s⁻³)"),
]:
    _ax.set_facecolor(LAND_COLOR)
    _im, _ = plot_global_field(
        _ax, _xk, _yk, _arr, "frontogenesis_tendency", CMAP_CFG,
        diverging_cmaps=DIVERGING, add_coastline=False)
    if _im is not None:
        plt.colorbar(_im, ax=_ax, fraction=0.04)
    _ax.set_title(_ttl, fontsize=11)
    _ax.set_xticks([])
    _ax.set_yticks([])
fig.suptitle("Frontogenesis definitions — identical frontal "
             "morphology, different weighting/units", fontsize=12)
plt.tight_layout()
plt.show()

### 6.3 Bachman et al. (2021) — log-magnitude PDFs, same region

Occurrence histograms of log₁₀|W*|, log₁₀|F|, log₁₀|∇b| over the
Kerguelen box, mimicking the reference's Figure 2 (F sits many
decades below W* and |∇b| by construction of its units).

In [ ]:
# Bachman et al. (2021) Fig-2-style comparison: overlaid log10
# magnitude histograms (occurrences) over the Kerguelen box.
def _bachman_pdfs(ax):
    """Overlaid log10-magnitude histograms (Bachman Fig 2 style).

    Inputs: ax (matplotlib axis).  Outputs: draws on ax.
    Generated by LH and Claude
    """
    panels = [
        ("log₁₀ |W*|", np.abs(_kerg("Wstar")[2]),
         "indianred"),
        ("log₁₀ |F| (Eq. 38)",
         np.abs(F_b38), "steelblue"),
        ("log₁₀ |∇b|", gradb_mag_kerg,
         "yellowgreen"),
    ]
    for name, vals, color in panels:
        v = vals[np.isfinite(vals)]
        v = np.log10(v[v > 0])
        ax.hist(v, bins=120, alpha=0.6, color=color, label=name)
    ax.set_ylabel("occurrences")
    ax.legend()


side_by_side(
    _bachman_pdfs,
    LIT_DIR / ("Wstar-F-gradb_Bachman-etal(2021)_"
               "Kerguelen-pdfs.png"),
    caption=("Compare modal positions and widths with Bachman et "
             "al. (2021) Fig 2.  |W*| and |∇b| compare "
             "directly; F uses their Eq. 38 (2Q·∇b/"
             "|∇b|, s⁻³).  Their fields are from "
             "an idealised channel run — expect lognormal "
             "SHAPES to match; modal offsets may be model/regime."),
)
plt.show()

## Summary — guard-style checks

Pass/fail checklist mirroring `dev/verify_subsets_real_data.py`:
finite fraction, plausible range, and channel-uniqueness guard on the
Gulf Stream domain.

In [ ]:
# Summary: quick stats + uniqueness guard (Gulf Stream domain).
seen = {}
print(f"{'field':24s} {'finite%':>8s} {'min':>11s} "
      f"{'max':>11s} {'unique':>7s}")
for ch in CHANNELS:
    sub = region_arrays[ch]["gulf_stream"][2]
    finite = np.isfinite(sub)
    frac = 100.0 * finite.mean()
    key = hash(sub[finite][::997].tobytes()) if finite.any() else ch
    dup = seen.get(key)
    seen[key] = ch
    print(f"{ch:24s} {frac:7.1f}% {np.nanmin(sub):11.3g} "
          f"{np.nanmax(sub):11.3g} {'DUP!' if dup else 'ok':>7s}")
    assert dup is None, f"{ch} identical to {dup}"
print("\nAll checks passed.")

In [ ]:
# Store-vs-live consistency: each final channel (store) must equal
# the function of its live-computed dependencies (Gulf Stream domain;
# land + halo-rim NaNs excluded).  This turns the two-path design
# (finals via RUN->store->LOAD, dependencies via live compute) into
# an explicit pass/fail test of the pipeline plumbing.
from dbof.preprocessing.physical_constants import (
    G, RHO0_REFERENCE, ALPHA, BETA,
)


def _gs(field):
    """Gulf Stream slice of one field.

    Inputs: field (str).  Outputs: 2D np.ndarray.
    Generated by LH and Claude
    """
    return region_arrays[field]["gulf_stream"][2]


CHECKS = {
    "ug = -(G/f) * dEta_dy":
        (_gs("ug"), -(G / _gs("coriolis_f")) * _gs("dEta_dy")),
    "vg = +(G/f) * dEta_dx":
        (_gs("vg"), (G / _gs("coriolis_f")) * _gs("dEta_dx")),
    "frontogenesis_ageo = tendency - geo":
        (_gs("frontogenesis_ageo"),
         _gs("frontogenesis_tendency") - _gs("frontogenesis_geo")),
}

REL_TOL = 1e-4       # float32 store vs float64 live recompute
for name, (store_v, recomputed) in CHECKS.items():
    m = np.isfinite(store_v) & np.isfinite(recomputed)
    scale = max(float(np.nanmax(np.abs(recomputed[m]))), 1e-300)
    rel = float(np.nanmax(np.abs(store_v[m] - recomputed[m]))) / scale
    flag = "OK  " if rel < REL_TOL else "FAIL"
    print(f"{flag} {name}  (max rel err {rel:.2e}, n={m.sum()})")
    assert rel < REL_TOL, name
print("\nStore-vs-live consistency: all checks passed.")

**Cross-references** — J machinery → `kinematic.ipynb`; b and |∇b|² (gradb2) → `frontal_structure.ipynb`; raw Eta/U/V → `native_fields.ipynb`.